In [1]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset
import matplotlib.pyplot as plt
import pandas as pd

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Plate parameters
size = 2.0  # Length of the plate
t_max = 5.0  # Maximum time

# Hyperparameters
layers = [3, 50, 50, 1]
lr = 0.001
batch_size = 32

Using device: cpu


In [3]:
!wget https://raw.githubusercontent.com/brcktn/Inverse-Heat-PINN/main/training_data/dual_gaussian.csv

class CSVDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file).values.astype('float32')
        self.X = torch.tensor(self.data[:, 1:4])
        self.y = torch.tensor(self.data[:, 4:])

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = CSVDataset('dual_gaussian.csv')
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

--2026-03-05 23:56:12--  https://raw.githubusercontent.com/brcktn/Inverse-Heat-PINN/main/training_data/dual_gaussian.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3662438 (3.5M) [text/plain]
Saving to: ‘dual_gaussian.csv’

dual_gaussian.csv   100%[===================>]   3.49M  16.5MB/s    in 0.2s    

2026-03-05 23:56:13 (16.5 MB/s) - ‘dual_gaussian.csv’ saved [3662438/3662438]



In [4]:
class HeatPINN(nn.Module):
    """
    A simple feedforward neural network
    """
    def __init__(self, layers):
        """
        Parameters
        ----------
        layers : list
            A list of integers specifying the number of neurons in each layer, including input and output layers.
        """
        super(HeatPINN, self).__init__()

        self.alpha = nn.Parameter(torch.tensor([0.1], dtype=torch.float32))  # Learnable thermal diffusivity

        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:
                self.layers.append(nn.Tanh())

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


model = HeatPINN(layers).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
def sample_points(num_points, size, t_max):
    """
    Sample points uniformly from the domain [-L/2, L/2] x [-L/2, L/2] x [0, t_max]

    Parameters
    ----------
    num_points : int
        The number of points to sample.
    size : float
        The size of the spatial domain (length of the plate).
    t_max : float
        The maximum time to sample from.

    Returns
    -------
    torch.Tensor
        A tensor of shape (num_points, 3) containing the sampled points.
    """
    x = torch.rand(num_points, 1) * size - size / 2  # Sample x in [-L/2, L/2]
    y = torch.rand(num_points, 1) * size - size / 2  # Sample y in [-L/2, L/2]
    t = torch.rand(num_points, 1) * t_max  # Sample t in [0, t_max]

    x.requires_grad = True
    y.requires_grad = True
    t.requires_grad = True

    return torch.cat([x, y, t], dim=1)


def sample_boundary_points(num_points, size, t_max):
    """
    Uniformly samples points from the boundary of the plate
    over the full time interval.

    Parameters
    ----------
    num_points : int
        The number of points to sample.
    size : float
        The size of the spatial domain (length of the plate).
    t_max : float
        The maximum time to sample from.

    Returns
    -------
    torch.Tensor
        A tensor of shape (num_points, 3) containing the sampled boundary points.
    """
    points_per_edge = num_points // 4
    # left edge (x = -L/2)
    x_left = torch.full((points_per_edge, 1), -size / 2)
    y_left = torch.rand(points_per_edge, 1) * size - size / 2
    t_left = torch.rand(points_per_edge, 1) * t_max
    # right edge (x = L/2)
    x_right = torch.full((points_per_edge, 1), size / 2)
    y_right = torch.rand(points_per_edge, 1) * size - size / 2
    t_right = torch.rand(points_per_edge, 1) * t_max
    # top edge (y = -L/2)
    x_top = torch.rand(points_per_edge, 1) * size - size / 2
    y_top = torch.full((points_per_edge, 1), -size / 2)
    t_top = torch.rand(points_per_edge, 1) * t_max
    # bottom edge (y = L/2)
    x_bottom = torch.rand(points_per_edge, 1) * size - size / 2
    y_bottom = torch.full((points_per_edge, 1), size / 2)
    t_bottom = torch.rand(points_per_edge, 1) * t_max

    # Concatenate all boundary points
    boundary_points = torch.cat([x_left, y_left, t_left,
                                  x_right, y_right, t_right,
                                  x_top, y_top, t_top,
                                  x_bottom, y_bottom, t_bottom], dim=0)
    
    # requires_grad not needed because no dirivatives will be used
    return boundary_points


sample_points(10, size, t_max)

tensor([[-0.8137, -0.6285,  4.6406],
        [-0.1519,  0.1164,  2.8384],
        [ 0.3115, -0.2604,  1.1194],
        [ 0.0443,  0.8221,  4.4436],
        [-0.2245, -0.1185,  3.2783],
        [ 0.8280,  0.1468,  0.3434],
        [-0.4077, -0.2865,  3.8544],
        [ 0.0202,  0.0170,  3.7984],
        [ 0.1340, -0.8321,  0.4718],
        [ 0.2728, -0.7773,  4.3829]], grad_fn=<CatBackward0>)